# PANDA training — configurable Kaggle wrapper

Clones the repo, installs deps, and runs `python -m src.train` on one fold.
Use this notebook for a single training run when you want weights for a new
Week 2 ensemble candidate.

**Inputs (attach on Kaggle):**
1. `panda-resized-train-data-512x512` (xhlulu)
2. No competition data needed if you use the preprocessed thumbnails

**Settings:** GPU T4 ON, Internet ON


In [ ]:
REPO = 'https://github.com/Shashaboii/AIMI_Panda_Challenge.git'
BRANCH = 'trackC'  # trackC contains the current Kaggle notebook + OOF updates
FOLD = 0
EPOCHS = 6
BACKBONE = 'efficientnet-b0'
LOSS = 'smoothl1'   # smoothl1, mse, ordinal
DROPOUT = 0.3
FEATURE_TAG = None  # e.g. 'tiles36_imsize192'


In [ ]:
import os
import subprocess

if os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/kaggle/working/repo', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/kaggle/working/repo'], check=True)
subprocess.run(['git', '-C', '/kaggle/working/repo', 'rev-parse', '--short', 'HEAD'], check=True)


In [ ]:
!pip install -q efficientnet_pytorch


In [ ]:
import glob

candidates = (
    glob.glob('/kaggle/input/*resized*512*/train_images/train_images')
    + glob.glob('/kaggle/input/*resized*512*/train_images')
    + glob.glob('/kaggle/input/**/train_images', recursive=True)
)

def png_count(path):
    try:
        return sum(name.lower().endswith('.png') for name in os.listdir(path))
    except FileNotFoundError:
        return 0

IMAGE_DIR = next((c for c in candidates if os.path.isdir(c) and png_count(c) > 5000), None)
if IMAGE_DIR is None:
    raise RuntimeError(
        'Could not find a PNG thumbnail dir. Attach the xhlulu resized dataset '
        'and do not use the raw competition TIFF directory.'
    )
print('IMAGE_DIR:', IMAGE_DIR, 'png files:', png_count(IMAGE_DIR))


In [ ]:
import subprocess
import sys

sys.path.insert(0, '/kaggle/working/repo')

def build_weight_name(fold):
    parts = [BACKBONE.replace('-', '')]
    if FEATURE_TAG:
        parts.append(FEATURE_TAG)
    if LOSS != 'smoothl1':
        parts.append(LOSS)
    parts.append(f'fold{fold}')
    return '_'.join(parts) + '.pth'

cmd = [
    'python', '-m', 'src.train',
    '--fold', str(FOLD),
    '--folds-csv', '/kaggle/working/repo/data/train_folds.csv',
    '--image-dir', IMAGE_DIR,
    '--epochs', str(EPOCHS),
    '--backbone', BACKBONE,
    '--loss', LOSS,
    '--dropout', str(DROPOUT),
    '--output-dir', '/kaggle/working',
]
if FEATURE_TAG is not None:
    cmd += ['--feature-tag', FEATURE_TAG]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd='/kaggle/working/repo', check=True)
print('Expected weight file:', build_weight_name(FOLD))


In [ ]:
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.pth'):
        print(f, os.path.getsize(f'/kaggle/working/{f}') / 1e6, 'MB')
